# Wardrobe Combinations — Gemini Image Generator

Generates photorealistic menswear images for all 154 wardrobe combinations using **Google Gemini image APIs**.

**No GPU required** — runs on any Colab runtime.

## Setup
1. Add your API key in Colab: **Secrets** (key icon in left sidebar) → add `GOOGLE_API_KEY`
2. Upload `wardrobe_mongo.json` when prompted
3. Run the model discovery cell to find available image models
4. Test, then batch-generate

In [ ]:
# Cell 1: Install dependencies
!pip install -q --upgrade google-genai Pillow tqdm

In [ ]:
# Cell 2: API key
import os
from getpass import getpass

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print('Loaded key from Colab Secrets')
except Exception:
    GOOGLE_API_KEY = getpass('Google API Key: ')
    print('Loaded key from manual input')

os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
print('Key set.')

In [ ]:
# Cell 3: Discover available image models
# Run this first to find which model names work on your account.
import requests

url = 'https://generativelanguage.googleapis.com/v1beta/models'
params = {'key': os.environ['GOOGLE_API_KEY'], 'pageSize': 100}
resp = requests.get(url, params=params)
resp.raise_for_status()
models = resp.json().get('models', [])

image_models = []
print('All models with image generation support:\n')
for m in models:
    methods = m.get('supportedGenerationMethods', [])
    name = m.get('name', '')
    display_name = m.get('displayName', '')
    if 'image' in name.lower() or 'imagen' in name.lower() or 'generateImages' in methods:
        image_models.append(m)
        print(f'  {name}')
        print(f'    Display: {display_name}')
        print(f'    Methods: {methods}')
        print()

if not image_models:
    print('No image models found. Listing all models with generateContent:\n')
    for m in models:
        methods = m.get('supportedGenerationMethods', [])
        if 'generateContent' in methods:
            print(f'  {m["name"]} - {methods}')

In [ ]:
# Cell 4: Upload wardrobe_mongo.json
import json
from google.colab import files

uploaded = files.upload()

with open('wardrobe_mongo.json', 'r') as f:
    combinations = json.load(f)

print(f'Loaded {len(combinations)} combinations')
print(f'Categories: {set(c["input_json"]["category"] for c in combinations)}')
print(f'Sample: {combinations[0]["combo_id"]}')

In [ ]:
# Cell 5: Prompt template

PROMPT_TEMPLATE = """Photorealistic premium commercial menswear photograph of an adult male model wearing a sophisticated business formal outfit.

MODEL & FRAMING
Adult male model with a natural, realistic physique. Face intentionally cropped out; do not show the full face. Frame the model from approximately the lower neck and chin area down to the shoes. Full outfit must remain visible. Upright, relaxed posture. Both hands placed naturally inside the trouser pockets. Legs positioned in a relaxed fashion-model stance: one leg crossed naturally in front of the other at the ankles. Natural human proportions and anatomy.

SHIRT
{shirt_color} ({shirt_hex}) long-sleeve dress shirt in fine cotton with subtle woven texture. Modern tailored fit. Semi-spread collar. Sleeves rolled neatly to just below the elbows. Top two buttons open, remaining buttons closed. Natural fabric texture and realistic folds. Preserve the specified color accurately under the lighting.

TROUSERS
{trouser_color} ({trouser_hex}) tailored formal trousers in smooth lightweight wool-blend suiting fabric. Modern tailored fit. Medium-high rise. Clean flat front. Natural tailoring and realistic fabric tension.

BELT
{belt_color} ({belt_hex}) simple leather dress belt with a simple rectangular brushed-metal buckle.

SHOES
{shoe_color} ({shoe_hex}) classic {shoe_type} dress shoes in full-grain polished leather with subtle natural reflections. Realistic leather texture.

WATCH
Stainless-steel dress wristwatch with a {dial_color} ({dial_hex}) dial and a {strap_color} ({strap_hex}) leather strap.

POSE
Relaxed, confident business-fashion posture. Both hands inside the trouser pockets. One leg crossed naturally in front of the other at the ankles. Natural weight distribution.

ENVIRONMENT
Upscale modern office interior. Contemporary understated architecture. Warm beige and taupe tones. Softly blurred background with realistic depth.

LIGHTING
Soft diffused indoor lighting. Warm neutral color temperature. Natural-looking shadows. Realistic highlights on leather, metal, and fabric.

CAMERA & PHOTOGRAPHY
Premium commercial menswear photography. DSLR photographic realism. Lens approximately 85mm. Straight-on perspective, slightly above waist height. Moderate shallow depth of field. Subject sharply focused. Background softly blurred. Vertical fashion editorial composition.

DO NOT include: blazer, jacket, tie, pocket square, jeans, sneakers, shorts, oversized clothing, exaggerated muscles, distorted anatomy, extra fingers, deformed hands, floating garments, unrealistic fabric folds, artificial-looking skin, excessive HDR, extreme contrast, dramatic fashion pose, visible full face, logos, brand names, watermark, text, cartoon, anime, illustration, painting, drawing."""


def assemble_prompt(input_json):
    return PROMPT_TEMPLATE.format(**input_json)


sample = combinations[0]['input_json']
prompt = assemble_prompt(sample)
print(f'Prompt length: {len(prompt)} chars, ~{len(prompt.split())} words')

In [ ]:
# Cell 6: Set the model to use
GEMINI_IMAGE_MODEL = 'gemini-3.1-flash-lite-image'

print(f'Using model: {GEMINI_IMAGE_MODEL}')

In [ ]:
# Cell 7: Generate a test image for L-001
import os
import base64
from google import genai
from IPython.display import display, HTML
from PIL import Image
from io import BytesIO

os.makedirs('/content/test_output', exist_ok=True)

client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])

test_combo = combinations[0]
prompt = assemble_prompt(test_combo['input_json'])
combo_id = test_combo['combo_id']

print(f'Generating test image for {combo_id} with {GEMINI_IMAGE_MODEL}...')

response = client.models.generate_content(
    model=GEMINI_IMAGE_MODEL,
    contents=prompt,
    config=genai.types.GenerateContentConfig(
        response_modalities=['IMAGE', 'TEXT'],
    ),
)

img_data = None
for part in response.candidates[0].content.parts:
    if part.inline_data and part.inline_data.mime_type.startswith('image/'):
        img_data = part.inline_data.data
        break

if img_data:
    path = f'/content/test_output/{combo_id}_gemini.png'
    with open(path, 'wb') as f:
        f.write(img_data)
    img = Image.open(path)
    img_thumb = img.copy()
    img_thumb.thumbnail((500, 750))
    buf = BytesIO()
    img_thumb.save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode()
    display(HTML(f'<img src="data:image/png;base64,{b64}" style="border:1px solid #ccc; border-radius:8px;"><br>{img.size[0]}x{img.size[1]}'))
    print(f'Saved: {path}')
else:
    print('No image returned. Response parts:')
    for part in response.candidates[0].content.parts:
        print(f'  type={type(part)}, text={getattr(part, "text", None)}')

---
## Batch Generation

If the test image above looks good, run the batch cell below to generate all 154 images.

In [ ]:
# Cell 8: Batch generate PENDING images only
# 110 images already generated — hardcoded below so no upload needed.
import os
import base64
import time
from tqdm.notebook import tqdm

ALREADY_DONE = {
    'D-020',
    'L-001', 'L-002', 'L-003', 'L-004', 'L-005', 'L-006', 'L-007', 'L-008', 'L-009', 'L-010',
    'L-011', 'L-012', 'L-013', 'L-014', 'L-015', 'L-016', 'L-017', 'L-018', 'L-019', 'L-020',
    'L-021', 'L-022', 'L-023', 'L-024', 'L-025', 'L-026', 'L-027', 'L-028', 'L-029', 'L-030',
    'L-031', 'L-032', 'L-033', 'L-034', 'L-035', 'L-036', 'L-037', 'L-038', 'L-039', 'L-040',
    'L-041', 'L-042', 'L-043', 'L-044', 'L-045', 'L-046', 'L-047', 'L-048', 'L-049', 'L-050',
    'L-051', 'L-052', 'L-053', 'L-054', 'L-055', 'L-056', 'L-057', 'L-058', 'L-059', 'L-060',
    'L-061', 'L-062', 'L-063', 'L-064', 'L-065', 'L-066', 'L-067', 'L-068', 'L-069', 'L-070',
    'L-071', 'L-072', 'L-073', 'L-074', 'L-075', 'L-076', 'L-077', 'L-078', 'L-079', 'L-080',
    'L-081', 'L-082', 'L-083', 'L-084', 'L-085', 'L-086', 'L-087', 'L-088', 'L-089',
    'P-001', 'P-002', 'P-003', 'P-004', 'P-005', 'P-006', 'P-007', 'P-008', 'P-009', 'P-010',
    'P-011', 'P-012', 'P-013', 'P-014', 'P-015', 'P-016', 'P-017', 'P-018', 'P-019', 'P-020',
}

os.makedirs('/content/output', exist_ok=True)

also_on_disk = set()
if os.path.isdir('/content/output'):
    for f in os.listdir('/content/output'):
        if f.endswith('.png'):
            also_on_disk.add(f.replace('.png', ''))

skip = ALREADY_DONE | also_on_disk
pending = [c for c in combinations if c['combo_id'] not in skip]
print(f'Total: {len(combinations)}, Already done: {len(skip)}, Pending: {len(pending)}')

generated = []
failed = []

for combo in tqdm(pending, desc='Generating pending images'):
    combo_id = combo['combo_id']
    path = f'/content/output/{combo_id}.png'

    prompt = assemble_prompt(combo['input_json'])

    try:
        response = client.models.generate_content(
            model=GEMINI_IMAGE_MODEL,
            contents=prompt,
            config=genai.types.GenerateContentConfig(
                response_modalities=['IMAGE', 'TEXT'],
            ),
        )
        for part in response.candidates[0].content.parts:
            if part.inline_data and part.inline_data.mime_type.startswith('image/'):
                with open(path, 'wb') as f:
                    f.write(part.inline_data.data)
                break

        generated.append(combo_id)
        time.sleep(1)

    except Exception as e:
        failed.append((combo_id, str(e)))
        print(f'FAILED {combo_id}: {e}')
        time.sleep(3)

print(f'\nDone! Generated: {len(generated)}, Failed: {len(failed)}')
if failed:
    print('Failed IDs:', [f[0] for f in failed])

In [ ]:
# Cell 9: Download newly generated images as zip
import zipfile
import os
from google.colab import files

zip_path = '/content/wardrobe_images.zip'
output_dir = '/content/output'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(output_dir)):
        if fname.endswith('.png'):
            zf.write(os.path.join(output_dir, fname), fname)

count = len([f for f in os.listdir(output_dir) if f.endswith('.png')])
print(f'Zipped {count} images')
files.download(zip_path)